# 02 - Data Cleaning
Cleans the raw data with the reusable functions in `src/data_cleaning.py`, validates the result,
saves it to `dataset/processed/` and writes a data-quality report to `reports/`.

**Cleaning policy**
| Issue | Action |
|---|---|
| Whitespace / empty text | trimmed / treated as missing |
| Non-numeric values in numeric columns | converted to missing |
| Popularity outside 0-100, 0-1 features outside 0-1, impossible loudness/duration/key/mode | row dropped |
| Tempo, time signature or year that is invalid | value set to missing, row kept |
| Missing track id/name/artist or missing core numeric value | row dropped |
| Missing genre | filled with `unknown` |
| Exact duplicates, then duplicate (track_id, track_genre) | removed (highest popularity kept) |
| Extreme durations | **kept** and flagged in `is_duration_outlier` |

In [ ]:
import sys, warnings
from pathlib import Path

# Make `src` importable whether Jupyter starts in the project root or in notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve_path

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

cfg = load_config()

## A. Load raw data

In [ ]:
from src.data_loader import find_raw_file, read_raw_csv, save_processed
from src.data_cleaning import clean_dataset, validate_clean, write_quality_report

raw_path = find_raw_file(cfg)
raw = read_raw_csv(raw_path, cfg["column_aliases"], cfg["loading"]["chunk_size"])
print(f"Raw rows: {len(raw):,}")

## B. Run the cleaning pipeline

In [ ]:
clean, report = clean_dataset(raw, cfg["cleaning"])
print(f"Clean rows: {len(clean):,}  (removed {report['rows_removed_total']:,})")
pd.DataFrame(report["steps"])

## C. What was invalid or missing?

In [ ]:
invalid = pd.Series(report["invalid_value_counts"], name="rows").to_frame()
invalid[invalid["rows"] > 0]

In [ ]:
print("Non-numeric values coerced to missing:", report["non_numeric_values_coerced_to_missing"] or "none")
print("Optional columns absent from this dataset:", report["absent_optional_columns"] or "none")
comparison = pd.DataFrame({"missing_raw": report["missing_before"], "missing_clean": report["missing_after"]}).fillna("n/a")
comparison

## D. Data types and the derived duration column

In [ ]:
clean[["duration_ms", "duration_min", "is_duration_outlier"]].describe().T

In [ ]:
clean.dtypes.to_frame('dtype')

## E. Validate the cleaned dataset
An empty list means every check passed.

In [ ]:
problems = validate_clean(clean)
print("Validation problems:", problems if problems else "none")
assert not problems, problems

## F. Save outputs

In [ ]:
outputs = save_processed(clean, cfg)
reports = write_quality_report(report, resolve_path(cfg["paths"]["reports_dir"]))
print("Cleaned CSV:    ", outputs["csv"])
print("Cleaned Parquet:", outputs["parquet"] or "skipped (pyarrow not installed)")
print("Quality report: ", reports["markdown"])

**Next:** `03_exploratory_data_analysis.ipynb`.